# Medical insurance cost

This dataset contains medical insurance cost information for 1338 individuals. It includes demographic and health-related variables such as age, sex, BMI, number of children, smoking status, and residential region in the US. The target variable is charges, which represents the medical insurance cost billed to the individual.


<hr />

## EDA

First, we're going to analize the behavior of each variable from the dataset in order to undestand the information presented



In [1]:
import pandas as pd

data = pd.read_csv('./data/MI/insurance.csv')

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [2]:
data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
## Quantity of missing values

missing = {
    'Number of Missing values: ': data.isna().sum(),
    'Percentage: ': data.isna().mean()
}

print('='*30)
print(missing)
print('='*30)

## Quantity of regions

{'Number of Missing values: ': age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64, 'Percentage: ': age         0.0
sex         0.0
bmi         0.0
children    0.0
smoker      0.0
region      0.0
charges     0.0
dtype: float64}


## Modelos

1. Random Forest Regressor
2. K Neighbors Regressor
3. Linear Regression
4. MLPRegresor

### Metricas de evaluacion
1. MAE
2. MSE
3. RMSE
4. R2
5. R2_ajustada
6. MAPE

<hr />

In [4]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error

from sklearn.model_selection import train_test_split
x = data[['age', 'sex', 'bmi', 'children', 'smoker', 'region']]
y = data['charges']

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=777)


### Linear Regressor

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np


## Adecuacion de caracteristicas (pipeline para hacerlo mas sencillo lol)
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('sex', OneHotEncoder(drop='first'), ['sex']),
        ('smoker', OneHotEncoder(drop='first'), ['smoker']),
        ('region', OneHotEncoder(drop='first'), ['region'])
    ],
    remainder='passthrough'
)


## Creacion del modelo y Pipeline
lr = Pipeline(steps=[
    ('preprocessor', preprocessor_lr),
    ('modelo', LinearRegression(n_jobs=-1))
])


## Entrenamiento y predicciones
lr.fit(x_train, y_train)
y_pred_lr = lr.predict(x_test)


## Metricas
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)
mape_lr = mean_absolute_percentage_error(y_test, y_pred_lr)

# 2. Calculo del R2 Ajustado
# n = numero de muestras en el set de prueba
# p = numero de características (predictores) independientes
n = x_test.shape[0]
p = x_test.shape[1] 

r2_ajustada_lr = 1 - ((1 - r2_lr) * (n - 1) / (n - p - 1))


print(f"MAE: {mae_lr:.4f}")
print(f"MSE: {mse_lr:.4f}")
print(f"RMSE: {rmse_lr:.4f}")
print(f"MAPE: {mape_lr:.4f} (o {mape_lr*100:.2f}%)")
print(f"R²: {r2_lr:.4f}")
print(f"R² Ajustadof: {r2_ajustada_lr:.4f}")


MAE: 4246.6217
MSE: 36205864.0465
RMSE: 6017.1309
MAPE: 0.4214 (o 42.14%)
R²: 0.7704
R² Ajustadof: 0.7651


### K Nearest Neighbors Regressor

In [9]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler

## Columnas a modificar
col_cat = ['sex', 'smoker', 'region']
col_num = ['age', 'bmi']

# Creacion de transformaciones a los datos
preprocessor_krn = ColumnTransformer(
    transformers=[
            ('col_cap_trans', OneHotEncoder(drop='first'), col_cat),
            ('col_num_trans', MinMaxScaler(), col_num)
        ],
        remainder='passthrough'
)

## Creacion del Pipeline para las transformaciones y modelos
knr = Pipeline(steps=[
    ('preprocessor', preprocessor_krn),
    ('modelo', KNeighborsRegressor(n_neighbors= 5, n_jobs= -1))
])

## Entrenamiento del modelo yy prediccion
knr.fit(x_train, y_train)
y_pred_knr = knr.predict(x_test)

## Metricas

mae_knr = mean_absolute_error(y_test, y_pred_knr)
mse_knr = mean_squared_error(y_test, y_pred_knr)
rmse_knr = root_mean_squared_error(y_test, y_pred_knr)
r2_knr = r2_score(y_test, y_pred_knr)
mape_knr = mean_absolute_percentage_error(y_test, y_pred_knr)

# 2. Calculo del R2 Ajustado
# n = numero de muestras en el set de prueba
# p = numero de características (predictores) independientes
n = x_test.shape[0]
p = x_test.shape[1] 

r2_ajustada_knr = 1 - ((1 - r2_knr) * (n - 1) / (n - p - 1))


print(f"MAE: {mae_knr:.4f}")
print(f"MSE: {mse_knr:.4f}")
print(f"RMSE: {rmse_knr:.4f}")
print(f"MAPE: {mape_knr:.4f} (o {mape_knr*100:.2f}%)")
print(f"R²: {r2_knr:.4f}")
print(f"R² Ajustado: {r2_ajustada_knr:.4f}") 

MAE: 4231.7049
MSE: 50069954.2968
RMSE: 7076.0126
MAPE: 0.3802 (o 38.02%)
R²: 0.6825
R² Ajustado: 0.6752


### Random Forest Regressor


In [10]:
from sklearn.ensemble import RandomForestRegressor

## Adecuacion de caracteristicas 
preprocessor_rfr = ColumnTransformer(
    transformers=[
        ('sex', OneHotEncoder(drop='first'), ['sex']),
        ('smoker', OneHotEncoder(drop='first'), ['smoker']),
        ('region', OneHotEncoder(drop='first'), ['region'])
    ],
    remainder='passthrough'
)

## Creacion del modelo
rfr = Pipeline(steps=[
    ('preprocessor', preprocessor_rfr),
    ('modelo', RandomForestRegressor(n_estimators= 500, random_state=777, n_jobs=-1))
])

## Entrenamiento del modelo y prediccion
rfr.fit(x_train, y_train)
y_pred_rfr = rfr.predict(x_test)

## Metricas
mae_rfr = mean_absolute_error(y_test, y_pred_rfr)
mse_rfr= mean_squared_error(y_test, y_pred_rfr)
rmse_rfr = root_mean_squared_error(y_test, y_pred_rfr)
r2_rfr = r2_score(y_test, y_pred_rfr)
mape_rfr = mean_absolute_percentage_error(y_test, y_pred_rfr)

# 2. Calculo del R2 Ajustado
# n = numero de muestras en el set de prueba
# p = numero de características (predictores) independientes
n = x_test.shape[0]
p = x_test.shape[1] 

r2_ajustada_rfr = 1 - ((1 - r2_rfr) * (n - 1) / (n - p - 1))


print(f"MAE: {mae_rfr:.4f}")
print(f"MSE: {mse_rfr:.4f}")
print(f"RMSE: {rmse_rfr:.4f}")
print(f"MAPE: {mape_rfr:.4f} (o {mape_rfr*100:.2f}%)")
print(f"R²: {r2_rfr:.4f}")
print(f"R² Ajustado: {r2_ajustada_rfr:.4f}") 

MAE: 2655.6293
MSE: 23307367.0980
RMSE: 4827.7704
MAPE: 0.3388 (o 33.88%)
R²: 0.8522
R² Ajustado: 0.8488


### MLPRegressor

In [ ]:
from sklearn.neural_network  import MLPRegressor


